# COVID-19 Data Analysis Project

Analyzing real-time COVID-19 data to identify patterns, trends, and correlations using Python.

In [ ]:
# Cell 1: Install required libraries (run this cell first)
import sys
!{sys.executable} -m pip install pandas numpy matplotlib seaborn scipy --quiet
print("Libraries installed successfully!")

In [ ]:
# Cell 2: Import Libraries and Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from datetime import datetime, timedelta

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print("All libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")

In [ ]:
# Cell 3: Data Collection - Load COVID-19 dataset from Our World in Data
# This is the most comprehensive global COVID-19 dataset
url = "https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv"

print("Downloading COVID-19 dataset... (this may take a moment)")
df = pd.read_csv(url)
print(f"Dataset loaded! Shape: {df.shape}")
print(f"Columns: {len(df.columns)}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")

In [ ]:
# Cell 4: Initial Data Exploration
print("First 5 rows:")
df.head()

In [ ]:
# Cell 5: Dataset Info
print("Dataset Info:")
df.info()

In [ ]:
# Cell 6: Basic Stats - Key Numerical Columns
key_cols = ['total_cases', 'new_cases', 'total_deaths', 'new_deaths', 
            'total_vaccinations', 'people_fully_vaccinated', 
            'reproduction_rate', 'icu_patients', 'hosp_patients']

available = [c for c in key_cols if c in df.columns]
df[available].describe()

In [ ]:
# Cell 7: Check Missing Values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Percentage (%)': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Percentage (%)', ascending=False)
print(f"Columns with missing values: {len(missing_df)}")
missing_df.head(20)

In [ ]:
# Cell 8: Data Cleaning - Filter and Prepare
# Keep only country-level data (not continent aggregates)
df_clean = df[df['continent'].notna()].copy()

# Convert date column
df_clean['date'] = pd.to_datetime(df_clean['date'])

# Fill missing numeric values with 0 for key metrics
fill_cols = ['new_cases', 'new_deaths', 'new_vaccinations', 
             'total_cases', 'total_deaths', 'total_vaccinations',
             'people_fully_vaccinated', 'icu_patients', 'hosp_patients']
for c in fill_cols:
    if c in df_clean.columns:
        df_clean[c] = df_clean[c].fillna(0)

# Create derived metrics
df_clean['death_rate'] = (df_clean['total_deaths'] / df_clean['total_cases'].replace(0, np.nan)) * 100
df_clean['cases_per_million'] = df_clean['total_cases'] / df_clean['population'] * 1_000_000
df_clean['deaths_per_million'] = df_clean['total_deaths'] / df_clean['population'] * 1_000_000

print(f"Cleaned dataset shape: {df_clean.shape}")
print(f"Countries: {df_clean['location'].nunique()}")
print(f"Date range: {df_clean['date'].min().date()} to {df_clean['date'].max().date()}")

In [ ]:
# Cell 9: Global Overview - Latest Global Numbers
latest = df_clean[df_clean['date'] == df_clean['date'].max()]

global_stats = {
    'Total Cases': f"{latest['total_cases'].sum():,.0f}",
    'Total Deaths': f"{latest['total_deaths'].sum():,.0f}",
    'Total Vaccinations': f"{latest['total_vaccinations'].sum():,.0f}",
    'People Fully Vaccinated': f"{latest['people_fully_vaccinated'].sum():,.0f}",
    'Global Death Rate': f"{latest['total_deaths'].sum() / latest['total_cases'].sum() * 100:.2f}%",
    'Total Countries Reporting': latest['location'].nunique()
}

print("=" * 50)
print("GLOBAL COVID-19 OVERVIEW")
print(f"As of: {df_clean['date'].max().date()}")
print("=" * 50)
for key, value in global_stats.items():
    print(f"{key:<30}: {value}")

In [ ]:
# Cell 10: Top 20 Most Affected Countries
latest = df_clean[df_clean['date'] == df_clean['date'].max()].copy()
top_cases = latest.nlargest(20, 'total_cases')[['location', 'total_cases', 'total_deaths', 'death_rate']]
top_cases.reset_index(drop=True, inplace=True)
top_cases.index = top_cases.index + 1
print("Top 20 Countries by Total Cases:")
top_cases

In [ ]:
# Cell 11: Global Trend - Cases Over Time
print("Computing global daily trends... (time-consuming cell)")
global_daily = df_clean.groupby('date').agg({
    'new_cases': 'sum',
    'new_deaths': 'sum',
    'new_vaccinations': 'sum'
}).reset_index()

global_daily['new_cases_7dma'] = global_daily['new_cases'].rolling(window=7).mean()
global_daily['new_deaths_7dma'] = global_daily['new_deaths'].rolling(window=7).mean()

fig, axes = plt.subplots(3, 1, figsize=(14, 12))

axes[0].fill_between(global_daily['date'], global_daily['new_cases'], alpha=0.3, color='blue')
axes[0].plot(global_daily['date'], global_daily['new_cases_7dma'], color='darkblue', linewidth=2)
axes[0].set_title('Global Daily New COVID-19 Cases (7-Day Moving Average)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('New Cases')

axes[1].fill_between(global_daily['date'], global_daily['new_deaths'], alpha=0.3, color='red')
axes[1].plot(global_daily['date'], global_daily['new_deaths_7dma'], color='darkred', linewidth=2)
axes[1].set_title('Global Daily New COVID-19 Deaths (7-Day Moving Average)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('New Deaths')

axes[2].fill_between(global_daily['date'], global_daily['new_vaccinations'] / 1e6, alpha=0.3, color='green')
axes[2].plot(global_daily['date'], global_daily['new_vaccinations'].rolling(window=7).mean() / 1e6, 
             color='darkgreen', linewidth=2)
axes[2].set_title('Global Daily New Vaccinations (7-Day Moving Average)', fontsize=13, fontweight='bold')
axes[2].set_ylabel('New Vaccinations (Millions)')
axes[2].set_xlabel('Date')

plt.tight_layout()
plt.show()

In [ ]:
# Cell 12: Top Countries Comparison - Cases and Deaths
top10 = latest.nlargest(10, 'total_cases')['location'].tolist()
top10_data = df_clean[df_clean['location'].isin(top10)]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for country in top10:
    cdata = top10_data[top10_data['location'] == country]
    axes[0].plot(cdata['date'], cdata['total_cases'] / 1e6, label=country, linewidth=1.5)

axes[0].set_title('Cumulative Cases by Country', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Total Cases (Millions)')
axes[0].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
axes[0].grid(True, alpha=0.3)

for country in top10:
    cdata = top10_data[top10_data['location'] == country]
    axes[1].plot(cdata['date'], cdata['total_deaths'] / 1e6, label=country, linewidth=1.5)

axes[1].set_title('Cumulative Deaths by Country', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Total Deaths (Millions)')
axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 13: Vaccination Progress and Impact
# Analyze vaccination rates vs death rates
print("Analyzing vaccination impact... (this may take a moment)")

vax_data = latest[
    (latest['people_fully_vaccinated'] > 0) & 
    (latest['total_cases'] > 1000) &
    (latest['population'] > 1_000_000)
].copy()

vax_data['vax_rate'] = (vax_data['people_fully_vaccinated'] / vax_data['population'] * 100).round(1)
vax_data['deaths_per_case'] = (vax_data['total_deaths'] / vax_data['total_cases'] * 100).round(2)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Top 10 vaccinated countries
top_vax = vax_data.nlargest(10, 'vax_rate')[['location', 'vax_rate', 'deaths_per_case']]
sns.barplot(data=top_vax, x='vax_rate', y='location', hue='location', 
            ax=axes[0], palette='viridis', legend=False)
axes[0].set_title('Top 10 Countries by Vaccination Rate (%)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Vaccination Rate (%)')

# Vaccination vs Death Rate scatter
scatter = axes[1].scatter(vax_data['vax_rate'], vax_data['deaths_per_case'],
                         c=vax_data['total_cases'], cmap='plasma', alpha=0.6, s=30)
axes[1].set_xlabel('Vaccination Rate (%)')
axes[1].set_ylabel('Death Rate (%)')
axes[1].set_title('Vaccination Rate vs Death Rate', fontsize=12, fontweight='bold')
plt.colorbar(scatter, ax=axes[1], label='Total Cases')

# Add trend line
mask = vax_data['vax_rate'].notna() & vax_data['deaths_per_case'].notna()
if mask.sum() > 2:
    slope, intercept, r_val, p_val, std_err = stats.linregress(
        vax_data.loc[mask, 'vax_rate'], vax_data.loc[mask, 'deaths_per_case']
    )
    x_line = np.linspace(vax_data['vax_rate'].min(), vax_data['vax_rate'].max(), 100)
    axes[1].plot(x_line, intercept + slope * x_line, 'r--', linewidth=1.5,
                 label=f'R={r_val:.3f}, p={p_val:.2e}')
    axes[1].legend(fontsize=9)

# Death rate distribution
sns.histplot(data=vax_data, x='deaths_per_case', bins=40, kde=True, ax=axes[2], color='coral')
axes[2].set_title('Distribution of Death Rates Across Countries', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Death Rate (%)')

plt.tight_layout()
plt.show()

In [ ]:
# Cell 14: Correlation Analysis
print("Correlation Analysis between key COVID-19 metrics...")

corr_vars = ['total_cases', 'total_deaths', 'total_vaccinations', 
             'people_fully_vaccinated', 'population', 'population_density',
             'median_age', 'aged_65_older', 'gdp_per_capita', 
             'extreme_poverty', 'cardiovasc_death_rate', 'diabetes_prevalence']

available_vars = [c for c in corr_vars if c in latest.columns]
corr_data = latest[available_vars].dropna()

if len(corr_data) > 5:
    corr_matrix = corr_data.corr()
    
    plt.figure(figsize=(12, 10))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
    sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='coolwarm', center=0,
                fmt='.2f', linewidths=0.5, square=True, 
                cbar_kws={'shrink': 0.8})
    plt.title('Correlation Matrix: COVID-19 Metrics & Socioeconomic Factors', 
              fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print("\nStrongest correlations with Total Cases:")
    case_corr = corr_matrix['total_cases'].drop('total_cases').sort_values(ascending=False)
    print(case_corr.round(3))
else:
    print("Insufficient data for correlation analysis.")

In [ ]:
# Cell 15: Heatmap - Monthly Cases by Continent
print("Generating monthly cases heatmap by continent... (time-consuming cell)")

df_clean['year_month'] = df_clean['date'].dt.to_period('M')
monthly_continent = df_clean.groupby(['year_month', 'continent'])['new_cases'].sum().reset_index()
monthly_pivot = monthly_continent.pivot(index='year_month', columns='continent', values='new_cases')
monthly_pivot = monthly_pivot.fillna(0) / 1e6

plt.figure(figsize=(16, 8))
sns.heatmap(monthly_pivot.T, cmap='YlOrRd', annot=False, 
            cbar_kws={'label': 'New Cases (Millions)'},
            linewidths=0.5, linecolor='white')
plt.title('Monthly COVID-19 Cases by Continent', fontsize=14, fontweight='bold')
plt.xlabel('Year-Month')
plt.ylabel('Continent')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 16: Infection Fatality Rate (IFR) Analysis
latest_ifr = latest[
    (latest['total_cases'] > 10000) & 
    (latest['continent'].notna())
].copy()

latest_ifr['IFR'] = (latest_ifr['total_deaths'] / latest_ifr['total_cases'] * 100).round(2)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# IFR by continent
ifr_by_cont = latest_ifr.groupby('continent')['IFR'].agg(['mean', 'std', 'count']).round(2)
ifr_by_cont.columns = ['Mean IFR (%)', 'Std Dev', 'Countries']

continent_order = latest_ifr.groupby('continent')['IFR'].mean().sort_values().index
sns.boxplot(data=latest_ifr, x='continent', y='IFR', order=continent_order, 
            palette='Set2', ax=axes[0])
axes[0].set_title('Infection Fatality Rate by Continent', fontsize=13, fontweight='bold')
axes[0].set_ylabel('IFR (%)')
axes[0].tick_params(axis='x', rotation=45)

print("IFR Statistics by Continent:")
print(ifr_by_cont)

# IFR vs Median Age
axes[1].scatter(latest_ifr['median_age'], latest_ifr['IFR'], 
               c=latest_ifr['total_cases'], cmap='viridis', alpha=0.6, s=30)
axes[1].set_xlabel('Median Age')
axes[1].set_ylabel('IFR (%)')
axes[1].set_title('IFR vs Median Age', fontsize=13, fontweight='bold')

mask = latest_ifr['median_age'].notna() & latest_ifr['IFR'].notna()
if mask.sum() > 2:
    slope, intercept, r_val, p_val, std_err = stats.linregress(
        latest_ifr.loc[mask, 'median_age'], latest_ifr.loc[mask, 'IFR']
    )
    x_line = np.linspace(latest_ifr['median_age'].min(), latest_ifr['median_age'].max(), 100)
    axes[1].plot(x_line, intercept + slope * x_line, 'r--', linewidth=1.5,
                 label=f'R={r_val:.3f}, p={p_val:.2e}')
    axes[1].legend(fontsize=9)

plt.colorbar(axes[1].collections[0], ax=axes[1], label='Total Cases')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 17: Reproduction Rate Over Time (Select Countries)
major_countries = ['United States', 'India', 'Brazil', 'United Kingdom', 
                   'Germany', 'France', 'Japan', 'South Korea']

rep_data = df_clean[df_clean['location'].isin(major_countries)]

plt.figure(figsize=(14, 7))
for country in major_countries:
    cdata = rep_data[rep_data['location'] == country].dropna(subset=['reproduction_rate'])
    if len(cdata) > 10:
        cdata = cdata[cdata['date'] >= '2020-03-01']
        plt.plot(cdata['date'], cdata['reproduction_rate'].rolling(14).mean(), 
                 label=country, linewidth=1.5)

plt.axhline(y=1.0, color='black', linestyle='--', alpha=0.5, label='R=1 (Threshold)')
plt.title('Reproduction Rate (R₀) Over Time - Major Countries (14-Day MA)', 
          fontsize=13, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Reproduction Rate')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 18: Key Findings and Conclusion
print("=" * 60)
print("KEY FINDINGS - COVID-19 DATA ANALYSIS")
print("=" * 60)

print("""
1. GLOBAL IMPACT:
   - The pandemic affected virtually every country worldwide
   - Cumulative cases and deaths show clear waves corresponding to variants

2. VACCINATION IMPACT:
   - Countries with higher vaccination rates generally show lower death rates
   - The inverse correlation suggests vaccines significantly reduced mortality

3. DEMOGRAPHIC FACTORS:
   - Median age shows positive correlation with IFR
   - Countries with older populations tended to have higher death rates

4. REGIONAL VARIATION:
   - Europe and Americas experienced multiple severe waves
   - Africa reported fewer cases per capita (potentially due to younger demographics
     and under-reporting)

5. TEMPORAL PATTERNS:
   - The 7-day moving average effectively smooths weekly reporting biases
   - Vaccine rollout in 2021 correlated with declining death rates
""")